In [79]:
from dotenv import load_dotenv
from anthropic import Anthropic 

load_dotenv() # Load environment vari
client  = Anthropic() # Initialize the Anthr
model = "claude-sonnet-4-5"


In [80]:
import httpx
client = Anthropic(http_client=httpx.Client(verify=False))

In [81]:
# helper func

def add_user_message(messages, txt):
    user_message = {"role": "user", "content": txt }
    messages.append(user_message)

In [82]:
def add_assistant_message(messages, txt):
    assistant_message = {"role": "assistant", "content": txt }
    messages.append(assistant_message)

Define the tool function

In [83]:
from datetime import datetime
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


#Ask Agent to help you generate the tool schema, this will be easier than you write your own one

1. name 
2. description
3. input_schema with properties

In [84]:

get_current_datetime_schema = ToolParam (
    {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}
)



In [85]:
current_time_test = get_current_datetime ("%Y-%m-%d %H:%M:%S")
current_time_test

'2026-07-29 11:23:15'

# Start requesting LLM with tool schema

In [86]:
messages = []

messages.append(
    {
        "role": "user",
        "content": "What is the current time now ? "

    }
)

response = client.messages.create(
    model = model,
    max_tokens=1000, 
    messages=messages,
    tools = [get_current_datetime_schema]

)

response

Message(id='msg_e88bbca5-0c2e-4a6c-b880-51fc226c3e18', container=None, content=[ToolUseBlock(id='tooluse_ph8QoYdUvJrcMtIEJ2bhow', caller=None, input={}, name='get_current_datetime', type='tool_use')], model='claude-4-5-sonnet', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo=None, input_tokens=607, output_tokens=38, output_tokens_details=None, server_tool_use=None, service_tier=None))

Message(id='msg_dd74f8a3-c670-4374-b93e-1003838df2a4', container=None, content=[ToolUseBlock(id='tooluse_LkKrdXU4YVvUCWklnt69uI', caller=None, input={}, name='get_current_datetime', type='tool_use')], model='claude-4-5-sonnet', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo=None, input_tokens=607, output_tokens=38, output_tokens_details=None, server_tool_use=None, service_tier=None))

In [87]:
response.content[0].input

{}

In [88]:
get_current_datetime(**response.content[0].input)

'2026-07-29 11:23:18'

In [89]:
messages

[{'role': 'user', 'content': 'What is the current time now ? '}]

In [90]:
messages.append(
    {
        "role" : "assistant", 
        "content": response.content
    }
)

In [91]:
messages

[{'role': 'user', 'content': 'What is the current time now ? '},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='tooluse_ph8QoYdUvJrcMtIEJ2bhow', caller=None, input={}, name='get_current_datetime', type='tool_use')]}]

In [ ]:
#assign result with get_current_datetime
result = get_current_datetime(**response.content[0].input)

messages.append(
    {
        "role" : "user", 
        "content": [
            {
                "tool_use_id": response.content[0].id,
                "type": "tool_result",
                "content": result,
                "is_error": False

            }
        ]
    }
)
'''
这些 key 名(type、tool_use_id、content、is_error)是 Anthropic Messages API 规定死的字段名,
不是你自己起的变量名。改成别的(比如 result、data、text)API 就识别不了,会报错。
API 服务端按这套固定字段名解析你发过去的 JSON。你把整个 messages 发给 API,服务端照着规范去找:

"type": "tool_result" → 哦这是工具结果 block
"tool_use_id" → 对应的是哪个 tool_use
"content" → 工具返回的内容
"is_error" → 这次工具是不是出错了(可选字段)
'''


In [93]:
response.content[0].id

'tooluse_ph8QoYdUvJrcMtIEJ2bhow'

In [94]:
messages

[{'role': 'user', 'content': 'What is the current time now ? '},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='tooluse_ph8QoYdUvJrcMtIEJ2bhow', caller=None, input={}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'tool_use_id': 'tooluse_ph8QoYdUvJrcMtIEJ2bhow',
    'type': 'tool_result',
    'content': '2026-07-29 11:23:18',
    'is_error': False}]}]

In [ ]:
final_answer = client.messages.create(
    model = model, 
    max_tokens= 1000,
    messages = messages,
    tools = [get_current_datetime_schema]

)
final_answer.content

Message(id='msg_27a88433-5111-4a37-86e7-6603bf2c9c6c', container=None, content=[TextBlock(citations=None, text='The current time is **11:23:18** on **July 29, 2026**.', type='text')], model='claude-4-5-sonnet', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo=None, input_tokens=673, output_tokens=24, output_tokens_details=None, server_tool_use=None, service_tier=None))

In [98]:
final_answer.content[0].text

'The current time is **11:23:18** on **July 29, 2026**.'

Message(id='msg_27a88433-5111-4a37-86e7-6603bf2c9c6c', container=None, content=[TextBlock(citations=None, text='The current time is **11:23:18** on **July 29, 2026**.', type='text')], model='claude-4-5-sonnet', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo=None, input_tokens=673, output_tokens=24, output_tokens_details=None, server_tool_use=None, service_tier=None))